In [1]:
# imports
import os, math, json, random
import torch
import torch.nn as nn
from torch.utils.data import random_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool
import torch.nn.functional as F
from itertools import product
from sklearn.model_selection import KFold
import numpy as np
import xgboost as xgb
from typing import Dict, Any, Tuple, List
import joblib  # optional, used to save fitted XGB in addition to .json

/home/kitao/anaconda3/envs/pp/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/kitao/anaconda3/envs/pp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/kitao/anaconda3/envs/pp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# dataset
NODE_FEATURE_DIM = 10
EDGE_FEATURE_DIM = 4

OP_TYPES = ['kn_input_op', 
            'kn_output_op', 
            'kn_add_op', 
            'kn_div_op',
            'kn_exp_op', 
            'kn_matmul_op', 
            'kn_mul_op',
            'kn_pow_op', 
            'kn_reduction_2_op', 
            'kn_sqrt_op']

def one_hot_op_type(op_type):
    out = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    out[OP_TYPES.index(op_type)] = 1.0
    return out

def compute_flops(op_type, input_tensors):
    if op_type == 'kn_matmul_op':
        M = input_tensors[0][-2]
        K = input_tensors[0][-1]
        N = input_tensors[1][-1]
        return 2 * M * N * K
    elif op_type in ['kn_add_op', 
                     'kn_div_op', 
                     'kn_exp_op', 
                     'kn_mul_op', 
                     'kn_pow_op', 
                     'kn_sqrt_op', 
                     'kn_reduction_2_op']:
        num_elements = 1
        for dim in input_tensors[0]:
            num_elements *= dim
        return num_elements
    elif op_type in ['kn_input_op', 'kn_output_op']:
        return 0
    else:
        raise ValueError(f"Unknown op type: {op_type}")

def get_node_features(node):
    # one-hot op type
    h_op_type = one_hot_op_type(node['op_type'])
    
    # in tensor dimensions
    in_tensors = []
    h_in_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['input_tensors']):
        in_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_in_dims[j + (i * 4)] = dim
    
    # out tensor dimensions
    out_tensors = []
    h_out_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['output_tensors']):
        out_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_out_dims[j + (i * 4)] = dim
    
    # in tensor sizes
    h_in_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(in_tensors):
        h_in_size[i] = math.prod(t)
    
    # out tensor sizes
    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    # computation cost in FLOPs
    flops = compute_flops(node['op_type'], in_tensors)
    h_flops = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    h_flops[0] = flops

    return torch.vstack([h_op_type, h_in_dims, h_out_dims, h_in_size, h_out_size, h_flops])

def get_edge_features(tensor):
    h_edge = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    for i, dim in enumerate(tensor['dim']):
        h_edge[i] = dim
    
    h_edge_size = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    h_edge_size[0] = math.prod(tensor['dim'])

    return torch.vstack([h_edge, h_edge_size])

class SubgraphDataset(Dataset):
    def __init__(self, root):   
        super().__init__(root)
        json_files = [f for f in os.listdir(root) if f.endswith('.json') and f.startswith('original_')]

        hash_to_time = json.load(open(os.path.join(root, 'performance.json'), 'r'))

        self.subgraph_files = []
        self.execution_times = []
        for json_file in json_files:
            hash = json_file[len('original_'):-len('.json')]
            if hash in hash_to_time:
                self.subgraph_files.append(json_file)
                self.execution_times.append(hash_to_time[hash])
        
        assert len(self.subgraph_files) == len(self.execution_times)

    def len(self):
        return len(self.subgraph_files)
    
    def get(self, idx):
        json_path = os.path.join(self.root, self.subgraph_files[idx])
        json_graph = json.load(open(json_path, 'r'))

        nodes = []
        edge_index = []
        edge_attr = []

        producer_of = {}

        for node_idx, node in enumerate(json_graph):
            nodes.append(get_node_features(node))

            for t in node['output_tensors']:
                producer_of[t['guid']] = node_idx

        for dst_idx, node in enumerate(json_graph):
            for t in node['input_tensors']:
                src_idx = producer_of[t['guid']]
                edge_index.append([src_idx, dst_idx])
                edge_attr.append(get_edge_features(t))
        
        return Data(
            x=torch.stack(nodes, dim=0),
            edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
            edge_attr=torch.stack(edge_attr, dim=0),
            y=torch.tensor([self.execution_times[idx]], dtype=torch.float),
        )

In [3]:
# models
class SubgraphEncoder(nn.Module):
    """
    Same as your original model up to the graph embedding 'g'.
    Produces a [num_graphs, hidden] representation.
    """
    def __init__(self, node_in=60, edge_in=8, hidden=256, layers=8, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_in = node_in
        self.edge_in = edge_in

        self.x_proj = nn.Sequential(
            nn.Linear(node_in, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
        )

        def make_mlp():
            return nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.ReLU(inplace=True),
                nn.Linear(hidden, hidden),
            )

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GINEConv(make_mlp(), edge_dim=edge_in))
        self.bns.append(nn.BatchNorm1d(hidden))
        for _ in range(layers - 1):
            self.convs.append(GINEConv(make_mlp(), edge_dim=edge_in))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden // 2, 1),
        )

    @torch.no_grad()
    def embed(self, data):
        """
        Convenience helper that runs in eval/no-grad and returns embeddings.
        """
        was_training = self.training
        self.eval()
        g = self.forward(data, return_embeddings=True)  # [B, hidden]
        if was_training:
            self.train()
        return g

    def forward(self, data, return_embeddings=False):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        # Flatten node features [N,6,10] -> [N,60] if needed
        if x.dim() == 3:
            x = x.view(x.size(0), -1)

        # Flatten edge features [E,2,4] -> [E,8] if needed
        if edge_attr is not None and edge_attr.dim() == 3:
            edge_attr = edge_attr.view(edge_attr.size(0), -1)

        x = self.x_proj(x)  # [N, hidden]

        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Graph-level representation
        if return_embeddings:
            return global_mean_pool(x, batch)  # [num_graphs, hidden]
        else:
            g = global_mean_pool(x, batch)  # [num_graphs, hidden]
            out = self.head(g).squeeze(-1)  # [num_graphs]
            return out

class SubgraphRegressorXGB(nn.Module):
    """
    Wraps the encoder and uses an XGBoost regressor as the head.
    - Call `fit_xgb(dataloader, y_tensor_or_extractor=...)` to train the XGB head.
    - Forward will return predictions if the XGB head is fitted; otherwise it returns embeddings.
    """
    def __init__(self, encoder, xgb_kwargs=None):
        super().__init__()
        self.encoder = encoder
        self.xgb = None
        self.xgb_kwargs = xgb_kwargs or dict(
            n_estimators=600,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
        )

    def forward(self, data):
        """
        If XGB is trained, returns predictions [num_graphs].
        If not, returns embeddings [num_graphs, hidden] so you can pretrain/diagnose.
        """
        g = self.encoder.embed(data)  # [B, hidden]
        if self.xgb is None:
            return g  # embeddings
        preds = self._xgb_predict_from_tensor(g)  # [B]
        return preds

    def _xgb_predict_from_tensor(self, g_tensor: torch.Tensor) -> torch.Tensor:
        g_cpu = g_tensor.detach().cpu().numpy()
        yhat = self.xgb.predict(g_cpu)
        return torch.from_numpy(yhat).to(g_tensor.device).float()

    @torch.no_grad()
    def extract_embeddings(self, dataloader, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Runs the encoder over a dataloader and returns (X, y) where:
          - X: np.ndarray of shape [num_graphs, hidden]
          - y: np.ndarray of shape [num_graphs]
        Assumes each batch item has `data.y` as the scalar regression target.
        """
        self.to(device)
        was_training = self.training
        self.eval()

        X, Y = [], []
        for batch in dataloader:
            batch = batch.to(device)
            g = self.encoder.embed(batch)  # [b, hidden]
            X.append(g.detach().cpu().numpy())
            if hasattr(batch, "y") and batch.y is not None:
                y = batch.y.view(-1).detach().cpu().numpy()
                Y.append(y)
        X = np.concatenate(X, axis=0)
        Y = np.concatenate(Y, axis=0) if len(Y) else None

        if was_training:
            self.train()
        return X, Y

    def fit_xgb(self, dataloader, device="cuda" if torch.cuda.is_available() else "cpu", X=None, y=None):
        """
        Fit the XGBoost head. You can either:
          - pass a dataloader (we'll extract embeddings + y), or
          - pass precomputed (X, y) numpy arrays.
        """
        if X is None or y is None:
            X, y = self.extract_embeddings(dataloader, device=device)
        self.xgb = xgb.XGBRegressor(**self.xgb_kwargs)
        self.xgb.fit(X, y)
        return self

    def predict_xgb(self, dataloader_or_tensor, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Inference helper:
          - If given a dataloader of pyg batches -> returns np.ndarray predictions for the whole set.
          - If given a torch.Tensor of embeddings -> returns np.ndarray predictions directly.
        """
        if self.xgb is None:
            raise RuntimeError("XGBoost head is not fitted. Call fit_xgb(...) first.")
        if isinstance(dataloader_or_tensor, torch.Tensor):
            return self.xgb.predict(dataloader_or_tensor.detach().cpu().numpy())

        # dataloader path
        self.to(device)
        was_training = self.training
        self.eval()

        preds = []
        for batch in dataloader_or_tensor:
            batch = batch.to(device)
            g = self.encoder.embed(batch)
            preds.append(self.xgb.predict(g.detach().cpu().numpy()))
        if was_training:
            self.train()
        return np.concatenate(preds, axis=0)

In [4]:
# common helpers
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _ensure_scalar_preds(model, batch, device):
    """
    Returns scalar predictions as a torch.FloatTensor [B].
    Works for:
      - Pure torch model that directly outputs [B]
      - XGB model where forward() returns embeddings if xgb is None
    """
    out = model(batch)  # could be [B] or [B, hidden]
    if isinstance(out, torch.Tensor):
        # If the model returned embeddings (2D), try to use XGB head if available
        if out.dim() == 2:
            if hasattr(model, "xgb") and model.xgb is not None:
                # run xgb on the embeddings
                g = out  # embeddings
                preds = model._xgb_predict_from_tensor(g)  # [B]
                return preds
            else:
                raise RuntimeError(
                    "Model returned embeddings (2D) but XGBoost head is not fitted yet. "
                    "Call model.fit_xgb(...) before evaluation."
                )
        # 1D tensor: good
        return out.view(-1).float()
    # Fallback (shouldn't happen): convert to tensor
    return torch.as_tensor(out, dtype=torch.float32, device=device).view(-1)


def evaluate(model, loader, device):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = _ensure_scalar_preds(model, batch, device)  # [B]
            all_pred.append(pred)
            all_true.append(batch.y.view(-1).to(device))

    if not all_pred:
        return float("nan"), float("nan"), float("nan")
    preds = torch.cat(all_pred)
    trues = torch.cat(all_true)
    mae = (preds - trues).abs().mean().item()
    rmse = torch.sqrt(((preds - trues) ** 2).mean()).item()
    return mae, mae, rmse  # (val_loss proxy, MAE, RMSE)


@torch.no_grad()
def pairwise_accuracy(preds: torch.Tensor, trues: torch.Tensor, exclude_ties: bool = True):
    n = preds.numel()
    if n < 2:
        return float('nan'), 0, 0
    pdiff = preds.view(n, 1) - preds.view(1, n)
    ydiff = trues.view(n, 1) - trues.view(1, n)
    tri_mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=preds.device), diagonal=1)
    comp_mask = tri_mask & (ydiff != 0) if exclude_ties else tri_mask
    total = int(comp_mask.sum().item())
    if total == 0:
        return float('nan'), 0, 0
    correct = ((torch.sign(pdiff) == torch.sign(ydiff)) & comp_mask).sum().item()
    return correct / total, correct, total

In [10]:
# GINE training
def train_one_gine(model, train_loader, val_loader, device, lr, weight_decay, max_epochs, ckpt_path):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5, verbose=False)

    best_val = float('inf')
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum, n_graphs = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            target = batch.y.view(-1).to(device)
            loss = F.mse_loss(pred, target)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            opt.step()

            train_loss_sum += loss.item() * batch.num_graphs
            n_graphs += batch.num_graphs

        train_mae = train_loss_sum / max(1, n_graphs)

        val_loss, val_mae, val_rmse = evaluate(model, val_loader, device)
        scheduler.step(val_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()
            torch.save(best_state, ckpt_path)

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val


def train_eval_grid_gine(
    data_root: str = "./data",
    batch_size: int = 16,
    weight_decay: float = 1e-5,
    max_epochs: int = 50,
    seed: int = 42,
    lr_grid = (1e-3,),
    hidden_grid = (64, 128, 256),
    layers_grid = (2, 3, 4),
    model_prefix: str = "cv",
    n_splits: int = 5,
):
    """
    KFold cross-validation over the full dataset.
    Grid-search over (hidden, layers, lr), selecting by OOF MAE.

    Returns best config, OOF metrics, per-fold metrics, and final all-data checkpoint.
    """
    # ---- setup ----
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- load dataset (full) ----
    dataset = SubgraphDataset(data_root)
    n_total = len(dataset)
    assert n_total >= max(3, n_splits), "Dataset too small for requested CV."

    # Pre-extract targets in dataset order
    scan_loader = DataLoader(dataset, batch_size=1, shuffle=False)
    ys = []
    with torch.no_grad():
        for batch in scan_loader:
            ys.append(batch.y.view(-1).item())
    y_all_np = np.array(ys, dtype=np.float32)
    y_all = torch.from_numpy(y_all_np)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # ---- grid over (hidden, layers, lr) with OOF bookkeeping ----
    best_cfg = None
    best_oof_mae = float("inf")
    all_results = []

    total_cfgs = len(hidden_grid) * len(layers_grid) * len(lr_grid)
    print(f"Running {n_splits}-fold CV over {total_cfgs} configs… (N={n_total})")

    for hidden, layers, lr in product(hidden_grid, layers_grid, lr_grid):
        print(f"\n=== Config: hidden={hidden}, layers={layers}, lr={lr} ===")
        oof_pred_np = np.zeros(n_total, dtype=np.float32)
        fold_metrics = []
        fold_ckpts = []

        # Stable tag for filenames from lr (e.g., 1e-03)
        lr_tag = (f"{lr:.0e}").replace("+", "")

        for fold_id, (train_idx, val_idx) in enumerate(kf.split(np.arange(n_total)), start=1):
            print(f"\n  Fold {fold_id}/{n_splits} | Train={len(train_idx)} Val={len(val_idx)}")

            model = SubgraphEncoder(node_in=60, edge_in=8, hidden=hidden, layers=layers, dropout=0.2).to(device)

            train_loader = DataLoader(torch.utils.data.Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
            val_loader   = DataLoader(torch.utils.data.Subset(dataset, val_idx),   batch_size=batch_size, shuffle=False)

            ckpt_path = f"{model_prefix}_h{hidden}_L{layers}_lr{lr_tag}_fold{fold_id}.pt"
            # train_one returns best val_loss proxy (unused here), and saves best weights to ckpt_path
            _ = train_one_gine(model, train_loader, val_loader, device, lr, weight_decay, max_epochs, ckpt_path)

            # Predict on validation fold (OOF slice)
            model.load_state_dict(torch.load(ckpt_path, map_location=device))
            model.eval()
            val_preds = []
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    pred = _ensure_scalar_preds(model, batch, device)  # [B]
                    val_preds.append(pred.detach().cpu())
            val_preds_t = torch.cat(val_preds) if len(val_preds) else torch.empty(0)
            val_preds_np = val_preds_t.numpy().astype(np.float32)

            # stash OOF preds
            oof_pred_np[val_idx] = val_preds_np

            # per-fold metrics (using your evaluate semantics)
            y_val_np = y_all_np[val_idx]
            y_val_t = torch.from_numpy(y_val_np)
            mae_fold = float(torch.mean(torch.abs(val_preds_t - y_val_t)).item())
            rmse_fold = float(torch.sqrt(torch.mean((val_preds_t - y_val_t) ** 2)).item())
            pacc_fold, pcorr, ptotal = pairwise_accuracy(val_preds_t, y_val_t, exclude_ties=True)

            fold_metrics.append({
                "fold": fold_id,
                "mae": mae_fold,
                "rmse": rmse_fold,
                "pairwise_acc": float(pacc_fold) if ptotal > 0 else float("nan"),
                "pairwise_correct": int(pcorr),
                "pairwise_total": int(ptotal),
            })
            print(
                f"    Fold {fold_id} -> MAE={mae_fold:.6f} | RMSE={rmse_fold:.6f} | "
                f"PairAcc={float(pacc_fold) if ptotal>0 else float('nan'):.4f} ({pcorr}/{ptotal})"
            )

            fold_ckpts.append(ckpt_path)

        # ---- OOF metrics for this config ----
        oof_pred_t = torch.from_numpy(oof_pred_np)
        mae_oof = float(torch.mean(torch.abs(oof_pred_t - y_all)).item())
        rmse_oof = float(torch.sqrt(torch.mean((oof_pred_t - y_all) ** 2)).item())
        pacc_oof, pcorr_oof, ptotal_oof = pairwise_accuracy(oof_pred_t, y_all, exclude_ties=True)

        print(
            f"\n  OOF -> MAE={mae_oof:.6f} | RMSE={rmse_oof:.6f} | "
            f"PairAcc={float(pacc_oof) if ptotal_oof>0 else float('nan'):.4f} ({pcorr_oof}/{ptotal_oof})"
        )

        cfg_result = {
            "config": {"hidden": hidden, "layers": layers, "dropout": 0.2, "lr": lr},
            "fold_metrics": fold_metrics,
            "oof_metrics": {
                "mae": mae_oof,
                "rmse": rmse_oof,
                "pairwise_acc": float(pacc_oof) if ptotal_oof > 0 else float("nan"),
                "pairwise_correct": int(pcorr_oof),
                "pairwise_total": int(ptotal_oof),
            },
            "oof_pred": oof_pred_np,      # numpy array
            "y_true": y_all_np,           # numpy array
            "fold_checkpoints": fold_ckpts,
        }
        all_results.append(cfg_result)

        if mae_oof < best_oof_mae:
            best_oof_mae = mae_oof
            best_cfg = {"hidden": hidden, "layers": layers, "dropout": 0.2, "lr": lr}

    assert best_cfg is not None, "Grid search did not evaluate any configuration."
    print(
        f"\n🏆 Best by OOF MAE: hidden={best_cfg['hidden']} layers={best_cfg['layers']} "
        f"lr={best_cfg['lr']} (OOF MAE={best_oof_mae:.6f})"
    )

    # ---- Train best config on ALL data and save final checkpoint (for deployment) ----
    # Tiny holdout from full data to drive scheduler; does not affect OOF metrics.
    idx_all = np.arange(n_total)
    split = max(1, int(0.95 * n_total))
    full_train_idx, tiny_val_idx = idx_all[:split], idx_all[split:]

    best_model = SubgraphEncoder(
        node_in=60, edge_in=8,
        hidden=best_cfg["hidden"],
        layers=best_cfg["layers"],
        dropout=best_cfg["dropout"],
    ).to(device)
    full_train_loader = DataLoader(torch.utils.data.Subset(dataset, full_train_idx), batch_size=batch_size, shuffle=True)
    tiny_val_loader   = DataLoader(torch.utils.data.Subset(dataset, tiny_val_idx),  batch_size=batch_size, shuffle=False)

    lr_tag = (f"{best_cfg['lr']:.0e}").replace("+", "")
    final_ckpt = f"{model_prefix}_best_full_lr{lr_tag}.pt"
    _ = train_one_gine(best_model, full_train_loader, tiny_val_loader, device,
                  best_cfg['lr'], weight_decay, max_epochs, ckpt_path=final_ckpt)
    torch.save(best_model.state_dict(), final_ckpt)
    print(f"Saved final all-data model: {final_ckpt}")

    # return best result + summary (keep a 'checkpoint' key for API parity)
    best_result = next(
        r for r in all_results
        if r["config"]["hidden"] == best_cfg["hidden"]
        and r["config"]["layers"] == best_cfg["layers"]
        and r["config"]["lr"] == best_cfg["lr"]
    )

    return {
        "best_config": best_cfg,
        "oof_metrics": best_result["oof_metrics"],
        "fold_metrics": best_result["fold_metrics"],
        "oof_pred": best_result["oof_pred"],
        "y_true": best_result["y_true"],
        "fold_checkpoints": best_result["fold_checkpoints"],
        "final_checkpoint": final_ckpt,
        "checkpoint": final_ckpt,     # for compatibility with previous callers
        "all_results": all_results,
    }

In [14]:
# XGB training
def train_eval_grid_xgb(
    encoder_ckpt: str,                     # path to the best encoder checkpoint from train_eval_grid
    encoder_cfg: Dict[str, Any],           # e.g. {"hidden": 256, "layers": 4, "dropout": 0.2}
    data_root: str = "./data",
    batch_size: int = 32,
    seed: int = 42,
    n_splits: int = 5,
    n_estimators_grid: Tuple[int, ...] = (200, 400, 800),
    max_depth_grid: Tuple[int, ...] = (4, 6, 8),
    learning_rate_grid: Tuple[float, ...] = (0.05,),
    model_prefix: str = "xgbcv",
) -> Dict[str, Any]:
    """
    Grid-search the XGB head (n_estimators, max_depth, learning_rate) using K-Fold OOF metrics,
    with a fixed pretrained encoder loaded from `encoder_ckpt`.
    Uses only SubgraphRegressorXGB.fit_xgb and predict_xgb; does not directly use xgb.XGBRegressor,
    and does not precompute embeddings via extract_embeddings in this function.
    Returns a dict with best config, OOF metrics, per-fold metrics, predictions,
    and saved model paths. The final fitted XGB on all data is returned and saved.
    """
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- 1) Build & load the fixed encoder -----------------------------------

    enc = SubgraphEncoder(
        node_in=60,
        edge_in=8,
        hidden=encoder_cfg.get("hidden", 256),
        layers=encoder_cfg.get("layers", 4),
        dropout=encoder_cfg.get("dropout", 0.2),
    ).to(device)
    enc.load_state_dict(torch.load(encoder_ckpt, map_location=device))
    enc.eval()

    # ---- 2) Load dataset and collect ground-truth targets in dataset order ----
    dataset = SubgraphDataset(data_root)
    n_total = len(dataset)
    assert n_total >= max(3, n_splits), "Dataset too small for the requested CV."

    # scan targets once in dataset order for OOF bookkeeping
    scan_loader = DataLoader(dataset, batch_size=1, shuffle=False)
    ys = []
    with torch.no_grad():
        for batch in scan_loader:
            ys.append(batch.y.view(-1).item())
    y_all = np.array(ys, dtype=np.float32)
    y_all_t = torch.from_numpy(y_all)

    # KFold splitter
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    best_cfg = None
    best_oof_mae = float("inf")
    all_results: List[Dict[str, Any]] = []

    total_cfgs = len(n_estimators_grid) * len(max_depth_grid) * len(learning_rate_grid)
    print(f"Running {n_splits}-fold CV for XGB over {total_cfgs} configs… (N={n_total})")

    # ---- 3) Grid over XGB params with OOF bookkeeping -------------------------
    for ne, md, lr in product(n_estimators_grid, max_depth_grid, learning_rate_grid):
        print(f"\n=== XGB Config: n_estimators={ne}, max_depth={md}, learning_rate={lr} ===")
        oof_pred = np.zeros(n_total, dtype=np.float32)
        fold_metrics = []
        fold_models = []

        # Stable tag for filenames from lr (e.g., 5e-02)
        lr_tag = (f"{lr:.0e}").replace("+", "")

        for fold_id, (tr_idx, va_idx) in enumerate(kf.split(np.arange(n_total)), start=1):
            print(f"\n  Fold {fold_id}/{n_splits} | Train={len(tr_idx)} Val={len(va_idx)}")

            # Fresh wrapper per fold with hyperparams set
            xgb_kwargs = dict(
                n_estimators=ne,
                max_depth=md,
                learning_rate=lr,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=seed + fold_id,
            )

            fold_model = SubgraphRegressorXGB(encoder=enc, xgb_kwargs=xgb_kwargs)

            train_subset = torch.utils.data.Subset(dataset, tr_idx)
            val_subset   = torch.utils.data.Subset(dataset, va_idx)
            train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
            val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False)

            # Fit XGB head using the wrapper (no direct xgb usage here)
            fold_model.fit_xgb(train_loader, device=device)

            # Predict on validation fold via the wrapper
            y_pred_va = fold_model.predict_xgb(val_loader, device=device).astype(np.float32)
            oof_pred[va_idx] = y_pred_va

            # per-fold metrics
            y_pred_va_t = torch.from_numpy(y_pred_va)
            y_va_t = torch.from_numpy(y_all[va_idx].astype(np.float32))
            mae_fold = float(torch.mean(torch.abs(y_pred_va_t - y_va_t)).item())
            rmse_fold = float(torch.sqrt(torch.mean((y_pred_va_t - y_va_t) ** 2)).item())
            pacc_fold, pcorr, ptotal = pairwise_accuracy(y_pred_va_t, y_va_t, exclude_ties=True)

            fold_metrics.append({
                "fold": fold_id,
                "mae": mae_fold,
                "rmse": rmse_fold,
                "pairwise_acc": float(pacc_fold) if ptotal > 0 else float("nan"),
                "pairwise_correct": int(pcorr),
                "pairwise_total": int(ptotal),
            })
            print(f"    Fold {fold_id} -> MAE={mae_fold:.6f} | RMSE={rmse_fold:.6f} | "
                  f"PairAcc={float(pacc_fold) if ptotal>0 else float('nan'):.4f} ({pcorr}/{ptotal})")

            # save fold head (xgb inside the wrapper)
            fold_json = f"{model_prefix}_ne{ne}_md{md}_lr{lr_tag}_fold{fold_id}.json"
            fold_model.xgb.save_model(fold_json)
            fold_pkl = f"{model_prefix}_ne{ne}_md{md}_lr{lr_tag}_fold{fold_id}.pkl"
            joblib.dump(fold_model.xgb, fold_pkl)
            fold_models.append({"json": fold_json, "pkl": fold_pkl})

        # OOF metrics for this XGB config
        oof_pred_t = torch.from_numpy(oof_pred)
        mae_oof = float(torch.mean(torch.abs(oof_pred_t - y_all_t)).item())
        rmse_oof = float(torch.sqrt(torch.mean((oof_pred_t - y_all_t) ** 2)).item())
        pacc_oof, pcorr_oof, ptotal_oof = pairwise_accuracy(oof_pred_t, y_all_t, exclude_ties=True)

        print(f"\n  OOF -> MAE={mae_oof:.6f} | RMSE={rmse_oof:.6f} | "
              f"PairAcc={float(pacc_oof) if ptotal_oof>0 else float('nan'):.4f} ({pcorr_oof}/{ptotal_oof})")

        cfg_result = {
            "config": {"n_estimators": ne, "max_depth": md, "learning_rate": lr},
            "fold_metrics": fold_metrics,
            "oof_metrics": {
                "mae": mae_oof,
                "rmse": rmse_oof,
                "pairwise_acc": float(pacc_oof) if ptotal_oof > 0 else float("nan"),
                "pairwise_correct": int(pcorr_oof),
                "pairwise_total": int(ptotal_oof),
            },
            "oof_pred": oof_pred,       # numpy array
            "y_true": y_all,            # numpy array
            "fold_models": fold_models, # list of saved per-fold model paths
        }
        all_results.append(cfg_result)

        if mae_oof < best_oof_mae:
            best_oof_mae = mae_oof
            best_cfg = {"n_estimators": ne, "max_depth": md, "learning_rate": lr}

    assert best_cfg is not None, "XGB grid search evaluated no configuration."
    print(f"\n🏆 Best XGB by OOF MAE: n_estimators={best_cfg['n_estimators']} "
          f"max_depth={best_cfg['max_depth']} learning_rate={best_cfg['learning_rate']} (OOF MAE={best_oof_mae:.6f})")

    # ---- 4) Fit best XGB on ALL data via wrapper and save ---------------------
    xgb_kwargs_final = dict(
        n_estimators=best_cfg["n_estimators"],
        max_depth=best_cfg["max_depth"],
        learning_rate=best_cfg["learning_rate"],
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=seed,
    )

    final_model = SubgraphRegressorXGB(encoder=enc, xgb_kwargs=xgb_kwargs_final)
    full_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    final_model.fit_xgb(full_loader, device=device)

    final_json = f"{model_prefix}_best_xgb.json"
    final_model.xgb.save_model(final_json)
    final_pkl = f"{model_prefix}_best_xgb.pkl"
    joblib.dump(final_model.xgb, final_pkl)
    print(f"Saved best XGB head: {final_json} (and {final_pkl})")

    # retrieve best_result for summary return
    best_result = next(
        r for r in all_results
        if r["config"]["n_estimators"] == best_cfg["n_estimators"]
        and r["config"]["max_depth"] == best_cfg["max_depth"]
        and r["config"]["learning_rate"] == best_cfg["learning_rate"]
)

    return {
        "best_xgb_config": best_cfg,
        "oof_metrics": best_result["oof_metrics"],
        "fold_metrics": best_result["fold_metrics"],
        "oof_pred": best_result["oof_pred"],
        "y_true": best_result["y_true"],
        "fold_models": best_result["fold_models"],
        "final_xgb_json": final_json,
        "final_xgb_joblib": final_pkl,
        "encoder_checkpoint": encoder_ckpt,
        "all_results": all_results,
    }

In [13]:
# training GINE encoder
encoder_results = train_eval_grid_gine(data_root='/home/kitao/projects/mirage/dataset/10_16_25_cleaned',
           model_prefix="models/10_29_exec_time_gine",
           lr_grid=(1e-3,),
           hidden_grid=(256,),
           layers_grid=(8,))

Running 5-fold CV over 1 configs… (N=233)

=== Config: hidden=256, layers=8, lr=0.001 ===

  Fold 1/5 | Train=186 Val=47
Epoch 001 | Train MAE: 0.0154 | Val MAE: 0.0897 | Val RMSE: 0.1265
Epoch 001 | Train MAE: 0.0154 | Val MAE: 0.0897 | Val RMSE: 0.1265
Epoch 010 | Train MAE: 0.0009 | Val MAE: 0.0200 | Val RMSE: 0.0264
Epoch 010 | Train MAE: 0.0009 | Val MAE: 0.0200 | Val RMSE: 0.0264
Epoch 020 | Train MAE: 0.0008 | Val MAE: 0.0149 | Val RMSE: 0.0195
Epoch 020 | Train MAE: 0.0008 | Val MAE: 0.0149 | Val RMSE: 0.0195
Epoch 030 | Train MAE: 0.0007 | Val MAE: 0.0122 | Val RMSE: 0.0193
Epoch 030 | Train MAE: 0.0007 | Val MAE: 0.0122 | Val RMSE: 0.0193
Epoch 040 | Train MAE: 0.0006 | Val MAE: 0.0139 | Val RMSE: 0.0219
Epoch 040 | Train MAE: 0.0006 | Val MAE: 0.0139 | Val RMSE: 0.0219
Epoch 050 | Train MAE: 0.0006 | Val MAE: 0.0126 | Val RMSE: 0.0203
    Fold 1 -> MAE=0.011799 | RMSE=0.018447 | PairAcc=0.7160 (774/1081)

  Fold 2/5 | Train=186 Val=47
Epoch 050 | Train MAE: 0.0006 | Val MAE:

In [15]:
# training XGB head
xgb_result = train_eval_grid_xgb(
    encoder_ckpt=encoder_results["final_checkpoint"],
    encoder_cfg=encoder_results["best_config"],
    data_root="/home/kitao/projects/mirage/dataset/10_16_25_cleaned",
    learning_rate_grid=(0.05,),
    n_estimators_grid=(300,),
    max_depth_grid=(6,),
    n_splits=5,
    model_prefix="models/10_29_exec_time_xgb",
)
print(xgb_result["best_xgb_config"], xgb_result["oof_metrics"])

Running 5-fold CV for XGB over 1 configs… (N=233)

=== XGB Config: n_estimators=300, max_depth=6, learning_rate=0.05 ===

  Fold 1/5 | Train=186 Val=47
    Fold 1 -> MAE=0.007096 | RMSE=0.013459 | PairAcc=0.7752 (838/1081)

  Fold 2/5 | Train=186 Val=47
    Fold 1 -> MAE=0.007096 | RMSE=0.013459 | PairAcc=0.7752 (838/1081)

  Fold 2/5 | Train=186 Val=47
    Fold 2 -> MAE=0.009853 | RMSE=0.021980 | PairAcc=0.8363 (904/1081)

  Fold 3/5 | Train=186 Val=47
    Fold 2 -> MAE=0.009853 | RMSE=0.021980 | PairAcc=0.8363 (904/1081)

  Fold 3/5 | Train=186 Val=47
    Fold 3 -> MAE=0.012574 | RMSE=0.020949 | PairAcc=0.8770 (948/1081)

  Fold 4/5 | Train=187 Val=46
    Fold 3 -> MAE=0.012574 | RMSE=0.020949 | PairAcc=0.8770 (948/1081)

  Fold 4/5 | Train=187 Val=46
    Fold 4 -> MAE=0.010408 | RMSE=0.018324 | PairAcc=0.8222 (851/1035)

  Fold 5/5 | Train=187 Val=46
    Fold 4 -> MAE=0.010408 | RMSE=0.018324 | PairAcc=0.8222 (851/1035)

  Fold 5/5 | Train=187 Val=46
    Fold 5 -> MAE=0.010130 | RMS